# 03 · Silver, DQS y quality gate

Fase CRISP-DM **Data Preparation / Evaluation**. Este notebook carga la Silver
ya generada por el pipeline y muestra:

1. Que hizo la limpieza (TransformationLog, no conjeturas)
2. Como se descompone el DQS en sus seis dimensiones
3. Que pasaria si una regla critica fallara
4. Las agregaciones Gold generadas con SQL sobre DuckDB

Requisito previo: `python scripts/run_pipeline.py`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

from src.utils.configloader import expand_path, load_catalogs, load_settings, load_sources

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

SETTINGS = load_settings()
from src.cleaning.transform_log import TransformationLog

from pathlib import Path

SILVER = ROOT / "data" / "silver"
print("Silver disponible:", SILVER.exists())
print([p.name for p in sorted(SILVER.glob("*.parquet"))] if SILVER.exists() else "ejecuta el pipeline")

## 3.1 · Que se limpio realmente: el TransformationLog

No se supone nada: cada operacion queda registrada con el numero de registros
afectados y ejemplos antes/despues.

In [ ]:
import json

runs = sorted((ROOT / 'artifacts' / 'runs').glob('run-*'), reverse=True)
print("Ultimo run:", runs[0].name if runs else "ninguno")
tlog_path = runs[0] / 'transform_log_onsv.json' if runs else None
if tlog_path and tlog_path.exists():
    tlog = json.loads(tlog_path.read_text(encoding='utf-8'))
    ops = tlog['operations']
    print(f'\n{len(ops)} operaciones registradas\n')
    resumen = pd.DataFrame([{
        'operacion': o['operation'],
        'columna': o.get('column'),
        'afectados': o.get('n_affected'),
        'estrategia': o.get('strategy'),
    } for o in ops])
    print(resumen.to_string(index=False))
else:
    print('No hay transform log: ejecuta el pipeline.')

## 3.2 · Silver: estructura y contrato

In [ ]:
from src.utils.configloader import load_contract

from src.validation.contract import validate_contract

sl = pd.read_parquet(SILVER / "onsv_silver_1.0.parquet")
print(f"Silver ONSV: {sl.shape[0]:,} filas x {sl.shape[1]} columnas\n")
print('Columnas canónicas:')
for i, c in enumerate(sl.columns, 1):
    print(f'  {i:2d}. {c:<28} {sl[c].dtype}')

In [ ]:
contract = load_contract("onsv", SETTINGS)
res = validate_contract(sl, contract)
print(f"\nContrato valido: {res['valid']}  (errores: {res['n_errors']})")
if res['errors']:
    for col, rule, msg in res['errors'][:10]:
        print(f'   {col}: {rule} -> {msg}')
print("\nColumnas extra toleradas (allow_extra):")
print('  ', res['details_by_column'].get('__extra_columns'))

## 3.3 · DQS descompuesto

El DQS es una media ponderada de seis dimensiones. Verlas por separado explica
mucho mas que el numero final.

In [ ]:
from src.quality import dimensions

from src.utils.configloader import load_catalogs

cd = load_contract("onsv", SETTINGS)
dqs = dimensions.compute_dqs(sl, SETTINGS, cd, load_catalogs("onsv"), dataset="onsv")
pesos = dqs['weights']
print('Dimension      Peso    Valor   Aporte')
print('-' * 46)
for k in pesos:
    print(f'{k:<14} {pesos[k]:.2f}  {dqs[k]:7.2f}  {dqs[k]*pesos[k]:7.2f}')
print('-' * 46)
print(f'{"DQS":<14} {"1.00":<6} {"":>7}  {dqs["dqs"]:7.2f}')
print(f'\nFilas evaluadas: {dqs["n_rows"]:,}')
print(f'Columna de frescura: {dqs["freshness_column"]} (ventana {dqs["freshness_window_days"]} d)')

### Por que la frescura es baja y eso es correcto

Los siniestros son de 2021-2025. Un registro de 2021 no es fresco en 2026: es un
dato historico. Por eso la frescura pesa 0.08 y el sistema lo dice en vez de

In [ ]:
print(f'Frescura: {dqs["freshness"]:.2f}')
print('Es una consecuencia de que la fuente es historica, no un defecto del proceso.')
print('Si la fuente se actualiza, el DQS sube solo, sin tocar configuracion.')

## 3.4 · Validacion de la correccion del DQS

Si `validity` aplicase `>= 0` a toda columna numerica, las coordenadas del Peru
(negativas) puntuarian 0%. Comprobamos que ya no es asi.

In [ ]:
rango_lat = (sl["latitud"].between(-18.5, 0.2)).mean() * 100
rango_lon = (sl["longitud"].between(-81.5, -68.5)).mean() * 100
print(f"Latitud dentro de rango:  {rango_lat:.2f}%")
print(f"Longitud dentro de rango: {rango_lon:.2f}%")
print(f"\n=> Un umbral >= 0 habria dado 0.00% en ambas. Correccion aplicada.")

## 3.5 · Features derivadas

In [ ]:
features = [c for c in sl.columns if c not in load_contract('onsv', SETTINGS)['contract']['columns']]
print('Features derivadas por src/transform/features.py:')
for f in features:
    print(f'   {f:<22} {sl[f].dtype}')

if 'tramo_dia' in sl.columns:
    print("\nSiniestros por tramo del dia:")
    print(sl['tramo_dia'].value_counts().to_string())

## 3.6 · Simular el fallo de una regla critica

El sistema esta disenado para detenerse. Inyectamos una violacion y comprobamos
que el gate la detecta y que el registro llega a cuarentena.

In [ ]:
from src.quality.domain_rules import detect_violations

from src.quality.gates import evaluate_gate

from src.utils.configloader import load_quality_rules

contaminado = sl.copy()
contaminado.loc[contaminado.index[:5], 'latitud'] = 99.0
print('Inyectadas 5 coordenadas fuera del rango peruano (99.0)\n')
viol = detect_violations(
    contaminado,
    load_quality_rules()['domain_rules']['onsv']['rules'],
    load_catalogs('onsv'),
)
crit = [v for v in viol if v['severity'] == 'critical']
print('Reglas criticas disparadas:')
for v in crit:
    print(f"   {v['rule']}: {v['count']} registros")

gate = evaluate_gate(
    dimensions.compute_dqs(contaminado, SETTINGS, cd, load_catalogs('onsv'), dataset='onsv'),
    viol,
    load_quality_rules(),
    'onsv',
    'demo-run',
)
print(f"\nGate resultante: {gate['status']}  (criticos: {gate['n_critical_errors']})")
print("\n=> El gate bloquea la publicacion en Gold model_ready.")

## 3.7 · Agregaciones Gold generadas con SQL

Declaradas en `config/quality/catalogs/*.yaml` y ejecutadas sobre DuckDB.

In [ ]:
from src.utils.configloader import load_aggregations

print('Agregaciones declaradas para onsv:')
for spec in load_aggregations('onsv'):
    val = f", value={spec['value']}" if 'value' in spec else ''
    print(f"   {spec['name']:<32} kind={spec['kind']:<9} by={spec['by']}{val}")

In [ ]:
agg = sorted((ROOT / 'reports' / 'quality' / 'aggregations').glob('onsv_*.json'))
if agg:
    latest = json.loads(agg[-1].read_text(encoding='utf-8'))
    for name, rows in latest['aggregations'].items():
        print(f'\n--- {name} ({len(rows)} filas) ---')
        print(pd.DataFrame(rows).head(8).to_string(index=False))
else:
    print('No hay agregaciones: ejecuta el pipeline.')

## 3.8 · Consulta SQL directa sobre la capa Silver

DuckDB lee los Parquet: no hay base de datos que mantener.

In [ ]:
import duckdb

silver_glob = str((ROOT / "artifacts" / "lineage" / "sipat_lineage.duckdb").resolve())
con = duckdb.connect(silver_glob, read_only=True)
try:
    print(con.execute(
        "SELECT dataset, layer, rows, cols FROM dataset_lines ORDER BY created_at DESC LIMIT 6"
    ).df().to_string(index=False))
finally:
    con.close()

In [ ]:
con = duckdb.connect()
try:
    sl_path = str((ROOT / 'data' / 'silver' / 'onsv_silver_1.0.parquet').resolve())
    out = con.execute(
        "SELECT departamento, COUNT(*) AS n, SUM(fallecidos) AS muertos "
        + f"FROM read_parquet('{sl_path}') "
        + "GROUP BY 1 ORDER BY muertos DESC, n DESC LIMIT 10"
    ).df()
    print(out.to_string(index=False))
finally:
    con.close()

## 3.9 · Conclusiones

1. La limpieza deja traza auditable en el TransformationLog.
2. El DQS se descompone y cada dimensión es explicable.
3. El gate detiene la publicación ante una regla critica (ver 3.6).
4. Las agregaciones son SQL en YAML, no codigo.
5. La capa Silver es consultable con SQL sin servidor adicional.